# Agentic AI Systems 4

#### Today:
- LLM in Python
- Common LLM Parameters
- Structured output
- Simple tools and simple chatbot
- Project introduction
  

Get your Groq API key and put it into an .env
https://console.groq.com/keys

### How to load an environmental variable:

In [ ]:
# load variables from .env
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")



# Grok example

In [ ]:
from groq import Groq

In [ ]:
client = Groq(
    api_key=GROQ_API_KEY,
)
print(client)

In [ ]:
# Use this to find out what models are available to you. Try to use a small model for your homework to save time. You can also use the model "openai/gpt-oss-20b" if you want to use a large model.

models = client.models.list()

for model in models.data:
    print(model.id)

In [ ]:
# get input from user
content = input("Ask a question from Groq: ")

messages=[
        {
            "role": "user",
            "content": content,
        }
    ]

model="openai/gpt-oss-20b"

chat_completion = client.chat.completions.create(
    model=model,
    messages=messages
)

print(chat_completion.choices[0].message.content)

### OpenAI example

In [ ]:
# load variables from .env
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [ ]:
from openai import AzureOpenAI

In [ ]:
# Ommitted for security reasons. Please replace with your own endpoint, model name, deployment name, and API version.
endpoint = ""
model_name = ""
deployment = ""
api_version = ""

In [ ]:
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=OPENAI_API_KEY,
)

In [ ]:
content = input("Ask a question from OpenAI: ")

messages=[
        {
            "role": "user",
            "content": content,
        }
    ]


response = client.chat.completions.create(
    messages=messages,
    model=deployment
)

print(response.choices[0].message.content)

## Let's use images with LLMs

In [ ]:
image_url ="../../../assets/redi_banner.png"

In [ ]:
# Open the image in jupyter:

from PIL import Image

image = Image.open(image_url)

print("Image format:", image.format)
print("Image size:", image.size)
print("Image mode:", image.mode)

In [ ]:
from IPython.display import display

display(image)

In [ ]:
# encode images to base64
import base64

def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
    return encoded_string

base64_image = encode_image_to_base64(image_url)

In [ ]:
print(base64_image)

In [ ]:
# Reuse from previous code

# Ommitted. Please replace with your own endpoint, model name, deployment name, and API version. You can find them at ai.azure.com in your foundry.
endpoint = ""
model_name = ""
deployment = ""
api_version = ""

In [ ]:

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=OPENAI_API_KEY,
)

In [ ]:
message = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Describe what you see in this image."
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{base64_image}"
                }
            }
        ]
    }
]

In [ ]:
# get response from the model
response = client.chat.completions.create(
    messages=message,
    model=deployment
)
print(response.choices[0].message.content)

## Structured Output

### JSON

In [ ]:
content = input("Ask a question from OpenAI: ")

messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. Always respond in JSON."
        },
        {
            "role": "user",
            "content": "Give me three popular programming languages and their main uses.",
        }
    ]


response = client.chat.completions.create(
    model=deployment,
    messages=messages,
    response_format={"type": "json_object"}
)

print(response.choices[0].message.content)

### Pydantic

In [40]:
from pydantic import BaseModel
from openai import OpenAI

In [41]:
# Define our expected output structure
class ProgrammingLanguage(BaseModel):
    name: str
    use: str


class LanguageResponse(BaseModel):
    languages: list[ProgrammingLanguage]



In [43]:
messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. Always respond in JSON."
        },
        {
            "role": "user",
            "content": "Give me three popular programming languages and their main uses.",
        }
    ]


response = client.chat.completions.parse(
    model=deployment,
    messages=messages,
    response_format=LanguageResponse
)

In [44]:
# Extract the parsed response
data = response.choices[0].message.parsed

print(data)

languages=[ProgrammingLanguage(name='Python', use='Web development, data analysis, machine learning, automation, and scripting'), ProgrammingLanguage(name='JavaScript', use='Interactive websites, frontend development, backend development with Node.js, and web applications'), ProgrammingLanguage(name='Java', use='Enterprise applications, Android development, backend systems, and large-scale software')]
